In [1]:
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd maxtext

Cloning into 'maxtext'...
remote: Enumerating objects: 96373, done.
remote: Counting objects: 100% (1774/1774), done.
remote: Compressing objects: 100% (880/880), done.
remote: Total 96373 (delta 1443), reused 907 (delta 894), pack-reused 94599 (from 3)
Receiving objects: 100% (96373/96373), 411.08 MiB | 19.40 MiB/s, done.
Resolving deltas: 100% (70375/70375), done.
/content/maxtext


In [2]:
!grep -R "dataset_type" .

./docs/tutorials/posttraining/dpo.md:    dataset_type=hf \
./docs/tutorials/posttraining/knowledge_distillation.md:  dataset_type=hf \
./docs/tutorials/posttraining/multimodal.md:    dataset_type=hf profiler=xplane
./docs/tutorials/pretraining.md:dataset_type=hf hf_path=allenai/c4 hf_data_dir=en train_split=train \
./docs/tutorials/pretraining.md:- `dataset_type`: `hf`
./docs/tutorials/pretraining.md:dataset_type=grain grain_file_type=arrayrecord grain_train_files=/tmp/gcsfuse/array-record/c4/en/3.0.1/c4-train.array_record* grain_worker_count=2 \
./docs/tutorials/pretraining.md:- `dataset_type`: `grain`
./docs/tutorials/pretraining.md:dataset_type=tfds dataset_path=gs://<DATASET_PATH> dataset_name='c4/en:3.0.1' train_split=train \
./docs/tutorials/pretraining.md:- `dataset_type`: `tfds`
./docs/run_maxtext/run_maxtext_single_host_gpu.md:dataset_type: Synthetic or pass a real bucket
./docs/run_maxtext/run_maxtext_single_host_gpu.md:  dataset_type=synthetic enable_checkpointing=True steps

In [40]:
!sed -n '1,200p' docs/guides/data_input_pipeline/data_input_hf.md

# Hugging Face pipeline

The Hugging Face pipeline supports streaming directly from the Hugging Face Hub, or from a Cloud Storage bucket in Hugging Face supported formats (parquet, json, etc.). This is through the Hugging Face [`datasets.load_dataset` API](https://huggingface.co/docs/datasets/en/loading) with `streaming=True`, which takes in `hf_*` parameters.

## Example config for streaming from Hugging Face Hub (no download needed)

In [`src/maxtext/configs/base.yml`](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/configs/base.yml) or through command line, set the following parameters:

```yaml
dataset_type: hf
hf_path: 'allenai/c4'  # for using https://huggingface.co/datasets/allenai/c4
hf_data_dir: 'en'
hf_train_files: ''
# set eval_interval > 0 to use the specified eval dataset, otherwise, only metrics on the train set will be calculated.
eval_interval: 10000
hf_eval_split: 'validation'
hf_eval_files: ''
# for HF pipeline, tokenizer_path can be a path in Huggin

In [41]:
!sed -n '1,200p' docs/guides/data_input_pipeline/data_input_tfds.md

# TFDS pipeline

1. Download the Allenai C4 dataset in TFRecord format to a Cloud Storage bucket. For information about cost, see [this discussion](https://github.com/allenai/allennlp/discussions/5056)

```shell
bash download_dataset.sh {GCS_PROJECT} {GCS_BUCKET_NAME}
```

2. In [`src/maxtext/configs/base.yml`](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/configs/base.yml) or through command line, set the following parameters:

```yaml
dataset_type: tfds
dataset_name: 'c4/en:3.0.1'
# set eval_interval > 0 to use the specified eval dataset. Otherwise, only metrics on the train set will be calculated.
eval_interval: 10000
eval_dataset_name: 'c4/en:3.0.1'
eval_split: 'validation'
# TFDS input pipeline only supports tokenizer in spm format
tokenizer_path: 'src/maxtext/assets/tokenizers/tokenizer.llama2'
```


In [42]:
!sed -n '1,250p' docs/guides/data_input_pipeline/data_input_grain.md

# Grain pipeline

## The recommended input pipeline for determinism and resilience!

[Grain](https://google-grain.readthedocs.io/en/latest/) is a library for reading data for training and evaluating JAX models. It’s designed to be:

- **Powerful**: Users can bring arbitrary Python transformations.
- **Flexible**: Users can readily override Grain components for their needs.
- **Deterministic**: Multiple runs of the same pipeline will produce the same outputs.
- **Resilient to preemptions**: With minimal-sized checkpoints, users can resume the dataloader from the point at which it was preempted and produce the same output as if it was never preempted.
- **Performant**: Achieved with multiprocessing with shared memory. Tested on multiple data modalities.
- **With minimal dependencies**: Does not depend on ML frameworks (Tensorflow).

## Why is determinism important for a data input pipeline?

Determinism in a data input pipeline means that the same input data always results in the same se

In [43]:
!sed -n '1,250p' docs/guides/data_input_pipeline/olmo_grain.md

# OLMo numpy pipeline (`dataset_type=olmo_grain`)

Grain-based input pipeline for AI2's pre-tokenized OLMo data mixes (e.g.
`OLMo-mix-0925-official.txt`). Reads headerless flat `.npy` token streams
from a gcsfuse mount, shards across hosts, optionally masks repeated-n-gram
instances, and yields the shapes the MaxText pretrain trainer expects.

## Quick start

1. **Download the data** to a GCS bucket. `--mix-file` is a local AI2 manifest listing relative npy paths to fetch from AI2's public bucket (e.g. `OLMo-mix-0925-official.txt` for the 6T pretrain mix or `OLMo-midtraining-mix-0625-100B.txt` for the 100B midtraining mix).

   ```bash
   python tools/data_generation/download_olmo_data_to_gcs.py \
       --mix-file ./OLMo-mix-0925-official.txt \
       --gcs-dest gs://my-bucket/dataset/ \
       --staging-dir /mnt/local-ssd/olmo-staging \
       --workers 16
   ```

2. **Mount it read-only** with gcsfuse (`np.memmap` needs a local path):

   ```bash
   gcsfuse --implicit-dirs --o ro my